# M3-CLV 축별 적응형 가치그래프 — Dunnhumby seed 42 validation

M1, 같은 설정의 V-only(β_N=0), 전체 M3-CLV를 비교합니다. 거래활동 수준은 **다음 거래와 신규상품 확장으로 이어지는 N 관계**를, 거래당 가치 수준은 **장바구니 금액 기여 V 관계**를 각각 조절합니다. N 관계는 사용자 효과를 제거하고 카테고리 평균으로 축소한 아이템 수준 추정치입니다. test와 holdout은 만들거나 평가하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess

REVIEWED_SHA = 'TO_BE_PINNED'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json, torch
from lightgcn_clv_m3_axis_adaptive import (
    configure_m3_axis_adaptive_dunnhumby_run,
    preflight_summary,
    run_experiment,
)

cfg = configure_m3_axis_adaptive_dunnhumby_run()
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
result_df = run_experiment(cfg)

In [ ]:
from IPython.display import display

columns = [
    'model_id', 'role',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'revenue@10', 'revenue@20', 'revenue@50', 'arp@10',
    'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
    'eff_catalog@10', 'top10_share@10', 'top100_share@10',
    'value_alignment',
]
available = [column for column in columns if column in result_df.columns]
validation = result_df[result_df['split'].eq('val')][available]
display(validation.sort_values('model_id'))
print('screening 판정:')
print(json.dumps(result_df.attrs['screening_decision'], ensure_ascii=False, indent=2))
print('결과 파일:', result_df.attrs['result_paths'])